In [1]:
# ============================
# LSTM 中文情感分析
# 已适配你的文件路径：C:\2\data\weibo_senti_100k.csv
# 直接复制运行！
# ============================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import jieba
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import csv

# ===================== 1. 配置 =====================
class Config:
    def __init__(self):
        self.vocab_size = 10002
        self.embedding_size = 128
        self.hidden_size = 128
        self.num_layers = 2
        self.num_classes = 2
        self.pad_size = 32
        self.dropout = 0.5
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = 128
        self.lr = 0.001
        self.num_epochs = 8

cfg = Config()

# ===================== 2. 你的文件路径（正确！） =====================
data_path = r"C:\2\data\weibo_senti_100k.csv"
stop_path = r"C:\2\data\stopwords.txt"

# 加载停用词
stop_words = [line.strip() for line in open(stop_path, encoding='utf-8')]

# ===================== 3. 读取数据 & 生成词典 =====================
print("正在读取数据并生成词典...")
all_words = []
data_list = []

with open(data_path, encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) < 2:
            continue
        label = int(row[0])
        content = row[1]
        data_list.append((label, content))
        
        words = jieba.lcut(content)
        words = [w for w in words if w not in stop_words and len(w.strip()) > 0]
        all_words += words

# 构建词典
counter = Counter(all_words).most_common(10000)
word2id = {w:i+2 for i, (w, c) in enumerate(counter)}
word2id['<PAD>'] = 0
word2id['<UNK>'] = 1
print(f"词典完成，共 {len(word2id)} 个词")

# ===================== 4. 数据集 =====================
class WeiboSet(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        label, content = self.data[idx]
        words = jieba.lcut(content)
        ids = [word2id.get(w, 1) for w in words]
        
        if len(ids) < cfg.pad_size:
            ids += [0] * (cfg.pad_size - len(ids))
        else:
            ids = ids[:cfg.pad_size]
        
        return label, torch.tensor(ids, dtype=torch.long)

# 划分训练/测试
train_data = data_list[:int(len(data_list)*0.8)]
test_data = data_list[int(len(data_list)*0.8):]

train_loader = DataLoader(WeiboSet(train_data), batch_size=cfg.batch_size, shuffle=True)
test_loader = DataLoader(WeiboSet(test_data), batch_size=cfg.batch_size)

print(f"训练集：{len(train_data)} 条")
print(f"测试集：{len(test_data)} 条")

# ===================== 5. LSTM 模型 =====================
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.embedding_size)
        self.lstm = nn.LSTM(cfg.embedding_size, cfg.hidden_size, cfg.num_layers, batch_first=True, dropout=cfg.dropout)
        self.fc = nn.Linear(cfg.hidden_size, 2)
    
    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

model = LSTM().to(cfg.device)

# ===================== 6. 训练 =====================
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=cfg.lr)

print("\n开始训练...")
for epoch in range(cfg.num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for labels, seqs in train_loader:
        seqs, labels = seqs.to(cfg.device), labels.to(cfg.device)
        opt.zero_grad()
        out = model(seqs)
        loss = loss_fn(out, labels)
        loss.backward()
        opt.step()
        
        total_loss += loss.item()
        pred = torch.argmax(out, dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.3f} | Acc: {correct/total:.3f}")

# ===================== 7. 测试 =====================
def predict(text):
    model.eval()
    words = jieba.lcut(text)
    ids = [word2id.get(w,1) for w in words]
    if len(ids) < cfg.pad_size:
        ids += [0]*(cfg.pad_size-len(ids))
    else:
        ids = ids[:cfg.pad_size]
    with torch.no_grad():
        out = model(torch.tensor([ids]).to(cfg.device))
    return "正向" if torch.argmax(out) == 1 else "负向"

print("\n===== 测试结果 =====")
test_list = [
    "今天真开心！",
    "气死我了，太糟糕了",
    "这家店服务超级好，强烈推荐！",
    "东西很差，再也不买了"
]

for t in test_list:
    print(f"{t} → {predict(t)}")

print("\n✅ 模型运行成功！")

Building prefix dict from the default dictionary ...


正在读取数据并生成词典...


Dumping model to file cache C:\Users\叶湘伦\AppData\Local\Temp\jieba.cache
Loading model cost 0.408 seconds.
Prefix dict has been built successfully.


词典完成，共 10002 个词
训练集：95990 条
测试集：23998 条

开始训练...
Epoch 1 | Loss: 0.346 | Acc: 0.840
Epoch 2 | Loss: 0.249 | Acc: 0.892
Epoch 3 | Loss: 0.228 | Acc: 0.902
Epoch 4 | Loss: 0.211 | Acc: 0.911
Epoch 5 | Loss: 0.192 | Acc: 0.921
Epoch 6 | Loss: 0.172 | Acc: 0.932
Epoch 7 | Loss: 0.150 | Acc: 0.942
Epoch 8 | Loss: 0.130 | Acc: 0.950

===== 测试结果 =====
今天真开心！ → 正向
气死我了，太糟糕了 → 正向
这家店服务超级好，强烈推荐！ → 正向
东西很差，再也不买了 → 正向

✅ 模型运行成功！
